# SpecForge SDK Demo

This notebook demonstrates all the key features of the SpecForge Python SDK:
- Monitoring specifications against data
- Creating animations
- Exporting specifications
- Generating example data
- Checking satisfiability of specs and inline expressions
- Custom rendering in Jupyter

## Setup

First, let's import the SDK and set up our client.

In [ ]:
# Import the SpecForge SDK
from specforge_sdk import SpecForgeClient, flat_encoding, EXPORT_LILO, EXPORT_JSON
import pandas as pd
import json
import numpy as np

import os

# Get the port from environment variable or use default
port = os.environ.get("SPECFORGE_PORT", "8080")

# Initialize the client
specforgeClient = SpecForgeClient(base_url="http://localhost:" + port)

# Check connection
if specforgeClient.health_check():
    print(f"✓ Connected to SpecForge API v{specforgeClient.version()}")
else:
    print("✗ Cannot connect to SpecForge API")
    print("Make sure the SpecForge server is running on http://localhost:" + port)

## Data Preview

Let's look at our sample sensor data.

In [ ]:
# Load and preview the sensor data
data = pd.read_csv("sensor_data.csv")
print(f"Data shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
print(f"Time range: {data['time'].min():.1f} to {data['time'].max():.1f}")
print(
    f"Temperature range: {data['temperature'].min():.1f}°C to {data['temperature'].max():.1f}°C"
)

# Show first few rows
data.head(10)

In [ ]:
# Plot the data
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Temperature plot
ax1.plot(data["time"], data["temperature"], "b-", linewidth=2, label="Temperature")
ax1.axhline(y=10, color="r", linestyle="--", alpha=0.7, label="Min Safe (10°C)")
ax1.axhline(y=40, color="r", linestyle="--", alpha=0.7, label="Max Safe (40°C)")
ax1.set_ylabel("Temperature (°C)")
ax1.set_title("Temperature Over Time")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Humidity plot
ax2.plot(data["time"], data["humidity"], "g-", linewidth=2, label="Humidity")
ax2.axhline(y=20, color="orange", linestyle="--", alpha=0.7, label="Min Normal (20%)")
ax2.axhline(y=80, color="orange", linestyle="--", alpha=0.7, label="Max Normal (80%)")
ax2.set_xlabel("Time")
ax2.set_ylabel("Humidity (%)")
ax2.set_title("Humidity Over Time")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Monitoring with Custom Rendering

Let's monitor our temperature bounds specification against the sensor data. Using `return_json=True` (the default) enables SpecForge's custom visualization in Jupyter.

In [ ]:
# Monitor temperature bounds with custom rendering
specforgeClient.monitor(
    system="temperature_sensor",
    definition="always_in_bounds",
    data_file="sensor_data.csv",
    params_file="temperature_config.json",  # specifies the values of the minimum and maximum temperature params
    # return_timeseries=False
    # this is the default, so result is displayed by SpecForge notebook renderer.
)

The params can also be directly specified as a dictionary.

In [ ]:
temp_params = {"min_temperature": 10.0, "max_temperature": 24.0}
specforgeClient.monitor(
    system="temperature_sensor",
    definition="always_in_bounds",
    data_file="sensor_data.csv",
    params=temp_params,
)

# Monitor Different Specifications

Let's test different specifications from our file.

In [ ]:
# Test emergency condition specification
specforgeClient.monitor(
    system="temperature_sensor",
    definition="emergency_condition",
    data_file="sensor_data.csv",
    params=temp_params,
)

In [ ]:
# Test humidity correlation specification
specforgeClient.monitor(
    system="temperature_sensor",
    definition="humidity_correlation",
    data_file="sensor_data.csv",
    params=temp_params,
)

In [ ]:
# Test recovery specification
specforgeClient.monitor(
    system="temperature_sensor",
    definition="recovery_spec",
    data_file="sensor_data.csv",
    params=temp_params,
)

## Animations

You can animate your systems using the `animate` method.

In [ ]:
specforgeClient.animate(
    system="temperature_sensor",
    data_file="sensor_data.csv",
    svg_file="temp.svg",  # The SVG file to animate
    # return_gif=True         # Return base64 GIF string (optional)
    # save_gif="output.gif"   # Save the gif to a file (optional)
)

## Monitoring with Direct Data

Instead of using a CSV file, we can generate data programmatically and pass it directly to the monitoring function.

In [ ]:
# Generate synthetic sensor data using sine waves
import numpy as np

# Generate time points
time_points = np.linspace(0, 30, 100)  # 100 points from 0 to 30

# Generate temperature data (sine wave with some variation)
base_temp = 22.0  # Base temperature
temp_variation = 3.0  # Temperature variation amplitude
temperatures = (
    base_temp
    + temp_variation * np.sin(time_points * 0.3)
    + 0.5 * np.random.randn(len(time_points))
)

# Generate humidity data (different sine wave)
base_humidity = 45.0  # Base humidity
humidity_variation = 8.0  # Humidity variation amplitude
humidities = (
    base_humidity
    + humidity_variation * np.sin(time_points * 0.2 + 1.5)
    + np.random.randn(len(time_points))
)

# Create DataFrame - this is the pythonic way to work with timeseries
synthetic_df = pd.DataFrame(
    {"time": time_points, "temperature": temperatures, "humidity": humidities}
)

print(f"Generated {len(synthetic_df)} synthetic data points")
print(
    f"Temperature range: {synthetic_df['temperature'].min():.1f}°C to {synthetic_df['temperature'].max():.1f}°C"
)
print(
    f"Humidity range: {synthetic_df['humidity'].min():.1f}% to {synthetic_df['humidity'].max():.1f}%"
)

# Preview first few samples
print("\nFirst 5 samples:")
display(synthetic_df.head())

In [ ]:
# Plot the synthetic data
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Temperature plot
ax1.plot(time_points, temperatures, "b-", linewidth=2, label="Synthetic Temperature")
ax1.axhline(y=10, color="r", linestyle="--", alpha=0.7, label="Min Safe (10°C)")
ax1.axhline(y=24, color="r", linestyle="--", alpha=0.7, label="Max Safe (24°C)")
ax1.set_ylabel("Temperature (°C)")
ax1.set_title("Synthetic Temperature Over Time")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Humidity plot
ax2.plot(time_points, humidities, "g-", linewidth=2, label="Synthetic Humidity")
ax2.axhline(y=20, color="orange", linestyle="--", alpha=0.7, label="Min Normal (20%)")
ax2.axhline(y=80, color="orange", linestyle="--", alpha=0.7, label="Max Normal (80%)")
ax2.set_xlabel("Time")
ax2.set_ylabel("Humidity (%)")
ax2.set_title("Synthetic Humidity Over Time")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Monitor temperature bounds using direct DataFrame with custom rendering
specforgeClient.monitor(
    system="temperature_sensor",
    definition="always_in_bounds",
    data=synthetic_df,  # Using DataFrame directly
    encoding=flat_encoding(),  # Specify encoding for direct data
    params=temp_params,
)

In [ ]:
# Monitor humidity correlation using direct DataFrame
specforgeClient.monitor(
    system="temperature_sensor",
    definition="humidity_correlation",
    data=synthetic_df,  # Using DataFrame directly
    encoding=flat_encoding(),
    params=temp_params,
)

We can also return a pythonic timeseries, for further analysis using Python:

In [ ]:
# Get monitoring results as Python DataFrame for analysis
result_df = specforgeClient.monitor(
    system="temperature_sensor",
    definition="temperature_in_bounds",
    data=synthetic_df,
    encoding=flat_encoding(),
    params=temp_params,
    verdicts=False,
    return_timeseries=True,  # Return as DataFrame
)

if result_df is not None:
    print("✓ Direct data monitoring successful")
    print(f"Analyzed {len(result_df)} synthetic data points")

    if "value" in result_df.columns:
        percentage = result_df["value"].mean() * 100
        print(f"Temperature in bounds: {percentage:.1f}% of the time")

        # Show some results
        print("\nFirst 10 results:")
        display(result_df.head(10))
    else:
        print("Available columns:", list(result_df.columns))
        display(result_df.head(10))
else:
    print("✗ Direct data monitoring failed - no DataFrame returned")

## Robustness Analysis

SpecForge can compute robustness values that indicate how "close" a specification is to being violated.

In [ ]:
# Monitor with robustness analysis enabled
specforgeClient.monitor(
    system="temperature_sensor",
    definition="temperature_in_bounds",
    data_file="sensor_data.csv",
    params=temp_params,
    verdicts=False,
    robustness=True,  # Enable robustness computation
)

In [ ]:
# Get robustness results as DataFrame for analysis
robustness_df = specforgeClient.monitor(
    system="temperature_sensor",
    definition="temperature_in_bounds",
    data_file="sensor_data.csv",
    params=temp_params,
    robustness=True,  # Enable robustness computation
    verdicts=False,
    return_timeseries=True,  # Return as DataFrame
)

if robustness_df is not None:
    print("✓ Robustness analysis successful")
    print(f"Analyzed {len(robustness_df)} data points")

    if "robustness" in robustness_df.columns:
        # Compute average robustness value
        avg_robustness = robustness_df["robustness"].mean()
        print(f"Average robustness value: {avg_robustness:.3f}")

        # Show some statistics
        print(f"Min robustness: {robustness_df['robustness'].min():.3f}")
        print(f"Max robustness: {robustness_df['robustness'].max():.3f}")
        print(f"Std robustness: {robustness_df['robustness'].std():.3f}")

        # Show first few results
        print("\nFirst 10 robustness results:")
        display(robustness_df[["time", "value", "robustness"]].head(10))
    else:
        print("Available columns:", list(robustness_df.columns))
        print("Robustness column not found in results")
        display(robustness_df.head(10))
else:
    print("✗ Robustness analysis failed - no DataFrame returned")

## Verdict Analysis

Verdicts provide detailed information about why a specification is satisfied or violated.

In [ ]:
# Monitor with detailed verdicts (but no robustness for cleaner output)
specforgeClient.monitor(
    system="temperature_sensor",
    definition="humidity_correlation",
    data_file="sensor_data.csv",
    params=temp_params,
    verdicts=True,  # This is the default
    robustness=False,
)

If we turn verdicts off, the analysis tree is less helpful:

In [ ]:
# Monitor with detailed verdicts (but no robustness for cleaner output)
specforgeClient.monitor(
    system="temperature_sensor",
    definition="humidity_correlation",
    data_file="sensor_data.csv",
    params=temp_params,
    verdicts=False,
    robustness=False,
)

# Export Specifications

Export specifications to different formats. This is useful if you specification uses records, or many utility definitions. The export will turn into a single monolothic formula with no use of records.

In [ ]:
# Export to LILO format (native specification language)
lilo_result = specforgeClient.export(
    system="temperature_sensor",
    definition="always_in_bounds",
    export_type=EXPORT_LILO,
    return_string=True,  # Get the exported string
)

if lilo_result:
    print("✓ LILO export successful:")
    print(lilo_result)
else:
    print("✗ LILO export failed")

Optionally, the export can be configured using a param file or a dictionary. This will substitute the relevant variables with the values from the param.

In [ ]:
export1 = specforgeClient.export(
    system="temperature_sensor",
    definition="always_in_bounds",
    export_type=EXPORT_LILO,
    params_file="temperature_config.json",  # Use the same param file
    return_string=True,  # Get the exported string
)
print("export1:", export1)

export2 = specforgeClient.export(
    system="temperature_sensor",
    definition="always_in_bounds",
    export_type=EXPORT_LILO,
    params={"min_temperature": 10.0, "max_temperature": 24.0},
    return_string=True,  # Get the exported string
)

print("export2:", export2)

In [ ]:
specforgeClient.export(
    system="temperature_sensor",
    definition="humidity_correlation",
    export_type=EXPORT_JSON,
    # return_string=False is the default, so JSON is displayed
)

# Generate Example Data

Generate example data that satisfies our specifications.

In [ ]:
# Generate examples with custom rendering
specforgeClient.exemplify(
    system="temperature_sensor",
    definition="always_in_bounds",
    n_points=20,
    # return_timeseries=False is the default, so JSON is displayed
)

Exemplify commands can optionally be supplied with assumptions, as shown in the following example. If the 'rigidity' field is set to 'Hard', the assumption will be strictly enforced. If it is set to 'Soft', the assumption will be treated as a soft constraint, allowing for some violations, at the discretion of the solver.

In [ ]:
humidity_assumption = {"expression": "eventually (humidity > 25)", "rigidity": "Hard"}
specforgeClient.exemplify(
    system="temperature_sensor",
    definition="humidity_correlation",
    assumptions=[humidity_assumption],
    n_points=20,
)

They may also be supplied with a param file or a dictionary, which specifies some subset of the param variables. The unspecified variables will be solved for by the exemplifier.

In [ ]:
specforgeClient.exemplify(
    system="temperature_sensor",
    definition="humidity_correlation",
    assumptions=[{"expression": "always_in_bounds", "rigidity": "Hard"}],
    params={"min_temperature": 38.0},
    n_points=20,
)

In [ ]:
# Generate examples and get as DataFrame
example_df = specforgeClient.exemplify(
    system="temperature_sensor",
    definition="temperature_in_bounds",
    n_points=15,
    also_monitor=False,
    return_timeseries=True,  # Get as DataFrame
)

In [ ]:
if example_df is not None:
    print("✓ Generated example data as DataFrame:")
    display(example_df)

    # Plot the generated examples
    if "temperature" in example_df.columns and "humidity" in example_df.columns:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))

        ax1.plot(
            example_df["time"],
            example_df["temperature"],
            "bo-",
            label="Generated Temperature",
        )
        ax1.axhline(y=10, color="r", linestyle="--", alpha=0.7, label="Min Safe (10°C)")
        ax1.axhline(y=24, color="r", linestyle="--", alpha=0.7, label="Max Safe (24°C)")
        ax1.set_ylabel("Temperature (°C)")
        ax1.set_title("Generated Example Temperature Data")
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(
            example_df["time"],
            example_df["humidity"],
            "go-",
            label="Generated Humidity",
        )
        ax2.set_xlabel("Time")
        ax2.set_ylabel("Humidity (%)")
        ax2.set_title("Generated Example Humidity Data")
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
else:
    print("✗ Failed to generate example DataFrame")

# Satisfiability Checking

Check whether a specification, an inline expression, or the whole system is satisfiable. This is useful for catching contradictory requirements early.

In [ ]:
# Check satisfiability of a named spec
specforgeClient.check_satisfiability(
    system="temperature_sensor", definition="always_in_bounds", return_details=True
)

You can also check satisfiability of an inline expression, without needing to define it as a spec in the `.lilo` file.

In [ ]:
# Check satisfiability of an inline expression
specforgeClient.check_satisfiability(
    system="temperature_sensor",
    expression="always (min_temperature <= temperature && temperature <= max_temperature)",
    return_details=True,
)

In [ ]:
# An unsatisfiable expression: temperature can't be both above and below a threshold
specforgeClient.check_satisfiability(
    system="temperature_sensor",
    expression="temperature > max_temperature && temperature < min_temperature",
    return_details=True,
)

In [ ]:
# Simple boolean check (without details)
is_sat = specforgeClient.check_satisfiability(
    system="temperature_sensor", expression="eventually (temperature > max_temperature)"
)
print(f"Satisfiable: {is_sat}")